# Kata 26 — Validación y Retry con Error Feedback

> Spec: [`specs/026-validation-retry/spec.md`](../../specs/026-validation-retry/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

# Sonnet para mayor matiz en auto-corrección
client, settings = bootstrap(model="claude-sonnet-4-6", budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cuando una extracción falla validación, no la acepto ni reintento ciega.
Reintenta incluyendo el documento, la extracción fallida, y el error
**específico**. El modelo auto-corrige.

## 2. Por qué importa

Retry sin feedback es ruido — el modelo no sabe qué cambiar. Aceptar la
salida fallida rompe contratos. Reintentar cuando el dato no existe
fabrica.

## 3. Loop con feedback específico

In [ ]:
import json, re
from datetime import datetime

EXTRACT = {
    "name": "extract_event",
    "description": "Extrae detalles de un evento.",
    "input_schema": {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "date": {"type": "string", "description": "Formato YYYY-MM-DD"},
            "location": {"type": ["string", "null"]},
        },
        "required": ["title", "date"],
    },
}

def validate(extraction: dict) -> str | None:
    if "title" not in extraction or "date" not in extraction:
        return "missing required field"
    try:
        datetime.strptime(extraction["date"], "%Y-%m-%d")
    except ValueError:
        return f"date '{extraction['date']}' no es YYYY-MM-DD"
    return None

def extract_with_retry(client, doc, max_retries=2):
    last_err = None
    last_extraction = None
    for attempt in range(max_retries + 1):
        msg = doc
        if last_err:
            msg = (
                f"{doc}\n\nIntento previo falló validación: {last_err}\n"
                f"Output previo: {json.dumps(last_extraction, ensure_ascii=False)}\n"
                "Corrige sólo lo que el error señala. Mantén lo demás."
            )
        resp = client.messages.create(
            system="Extrae el evento. Fechas en formato YYYY-MM-DD estricto.",
            messages=[{"role":"user","content": msg}],
            tools=[EXTRACT],
            tool_choice={"type":"tool","name":"extract_event"},
        )
        last_extraction = next(b.input for b in resp.content if b.type == "tool_use")
        last_err = validate(last_extraction)
        if not last_err:
            return {"extraction": last_extraction, "attempts": attempt + 1, "ok": True}
    return {
        "extraction": last_extraction,
        "validation_error": last_err,
        "attempts": max_retries + 1,
        "ok": False,
        "needs_human_review": True,
    }

# Doc con fecha en formato no estricto — el modelo va a tender a normalizarla
doc = "Conferencia de IA, el 03/15/2026, en Bogotá."
print("=== retry con feedback ===")
result = extract_with_retry(client, doc)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 4. Anti-patrón — retry sin feedback específico

In [ ]:
def extract_dumb_retry(client, doc, max_retries=2):
    for attempt in range(max_retries + 1):
        resp = client.messages.create(
            system="Extrae el evento. Fechas en formato YYYY-MM-DD estricto.",
            messages=[{"role":"user","content": doc}],  # mismo prompt cada vez
            tools=[EXTRACT],
            tool_choice={"type":"tool","name":"extract_event"},
        )
        extraction = next(b.input for b in resp.content if b.type == "tool_use")
        err = validate(extraction)
        if not err:
            return {"extraction": extraction, "attempts": attempt + 1, "ok": True}
    return {"extraction": extraction, "validation_error": err, "attempts": max_retries + 1, "ok": False}

# Caso "dato no presente" — retry no debería ayudar pero el modelo puede inventar
doc_missing = "Hubo una reunión el martes pasado."
print("=== retry ciego (anti-patrón) ===")
print(json.dumps(extract_dumb_retry(client, doc_missing), indent=2, ensure_ascii=False))

print("\n=== retry con feedback (mismo doc, dato no presente) ===")
print(json.dumps(extract_with_retry(client, doc_missing), indent=2, ensure_ascii=False))

Para "el martes pasado" sin fecha absoluta, ningún retry resuelve: el
dato no está. El loop con feedback al menos lo deja con un mejor último
intento; el dumb retry puede oscilar entre fabricaciones.

## 5. Argumento de certificación

- **Retry recuperable**: error de formato/structure (fecha mal formateada).
  Feedback específico funciona.
- **Retry no recuperable**: dato ausente. Marcar `needs_human_review`,
  no insistir.
- **Feedback específico**: el error real, no "intenta de nuevo".
- Conexión: complementa Kata 5 (schema design), Kata 15 (cross-check),
  y dispara Kata 16 (handoff) cuando agota retries.

## 6. Auto-evaluación

**1. ¿Cuándo retry NO ayudará por más feedback que dé?**

Cuando el dato simplemente no está en la fuente. El modelo no puede
inventar y reintentar = invitación a alucinar.

**2. ¿Por qué el feedback debe ser específico y no "intenta de nuevo"?**

El modelo necesita saber **qué** cambiar. Sin el error específico, va a
producir variaciones aleatorias.

**3. ¿Qué pasa si tras 3 intentos sigue fallando?**

Marcar `needs_human_review` con la cadena de errores. Detenerse — más
intentos no resuelven y queman cuota.